# CP4 - Transformaciones II

### Como siempre, antes de empezar...

Si has cerrado el proyecto o has abierto `Nueva Ventana` en VS Code, pulsa en el primer icono del menú de la izquierda `Explorador` y después en `Abrir Carpeta`. 

Busca la ruta en la que guardas el repositorio de GitHub en tu PC, normalmente *`Documentos => GitHub => fundamentos_big_data` * y pulsa en `Abrir`. De esta forma, ya tendrás la carpeta de tu repositorio cargada en VS Code.

> ¡Importante! Antes de abrir esa carpeta, debes ir a GitHub Desktop y realizar un `pull` para asegurarte de que tus archivos están actualizados. También puedes hacer el pull desde VS Code.  

### Requisitos para arrancar el entonro
- Iniciar Docker (abrir docker desktop).
- Iniciar GitHub (abrir github desktop) y estar logueado.
- En el terminal, ejecutar el siguiente comando en la raíz del proyecto `fundamentos_big_data %` 
```bash
docker-compose up --build
```
> ¡NO TE OLVIDES! Docker debe estar abierto y en ejecución en tu pc para poder utilizarlo. 


### Seleccionar Kernel
- Seleccionar kernel
- Servidor de Juypter existente
- Escriba la dirección URL del servidor de Jupyter en ejecución
- Seleccionar http://127.0.0.1:8888/?token=xxxxxxxx y pulsar Enter
- 127.0.0.1 y pulsar Enter
- Seleccionar Python 3 (ipykernel)

___________________

## Objetivos de la sesión

- Trabajar con datos faltantes e incorrectos.
- Conocer y aplicar las acciones de Dataframes.
- Conocer y aplicar las agregaciones fundamentales.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window

# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("Pract2_PySpark") \
    .getOrCreate()

sc = spark.sparkContext

DATA_PATH = "/home/jovyan/work/data_practicas_1/"

### Añadir un nuevo dataset

Vamos a añadir un nuevo dataset. Accede a canvas y descarga el archivo `vuelos.zip`. Descomprímelo en tu ordenador y copia y pega la carpeta llamada "vuelos" a la carpeta `Documentos/GitHub/fundamentos_big_data/notebooks/data_practicas_1` (la carpeta en la que tenemos los datos). 

Asegúrate de haber hecho `pull` en GitHub. Una vez hayas pegado la carpeta de vuelos en la ruta que te comento, deberías hacer `commit` y `push` en GitHub. 

Más adelante cargaremos este dataset.

## 2 - Dataframes en PySpark

In [2]:
df4 = (spark.read.option('header', 'true').option('delimiter',',')
                     .csv(DATA_PATH + 'dataCSV.csv'))

### 2.7 Trabajar con datos Faltantes e Incorrectos

#### **Eliminar filas con algún valor `null`**

Lo primero que podemos hacer es hacer un recuento de las filas de DF y a continuación, compararlas con el número de filas resultantes, una vez eliminadas aquellas filas que contienen algún valor `null` en alguna de las columnas. Para ello, podemos usar la función `na`,`drop()` o `dropna()`.

In [3]:
print(f'Num filas df4: {df4.count()}')
print(f'Num filas df4 sin faltantes: {df4.na.drop().count()}')
print(f'Se han eliminado: {df4.count() - df4.na.drop().count()} filas.')

Num filas df4: 48137
Num filas df4 sin faltantes: 40390
Se han eliminado: 7747 filas.


In [4]:
print(f'Num elementos al quitarlos con dropna(): {df4.dropna().count()}')

Num elementos al quitarlos con dropna(): 40390


#### **Eliminar filas con `null` en columnas concretas**

Usamos el parámetro `subset=<[columnas]>` para seleccionar y eliminar filas para las que en una o varias columnas seleccionadas, tenemos algún valor `null`.

In [5]:
print(f'Num filas df4: {df4.count()}')
null_likes = df4.na.drop(subset=['likes']).count()
print(f'Num filas df4 sin faltantes en "Likes": {null_likes}')

Num filas df4: 48137
Num filas df4 sin faltantes en "Likes": 41043


#### **Imputación de datos**

Para llevar a cabo la imputación, usaremos la función `fillna()` y le pasaremos como parámetro el valor por el que queremos realizar la imputación.

In [6]:
df4.select(
    F.col('likes'),
    F.col('dislikes')
).orderBy(F.col('likes')).show(10)

+-----+--------+
|likes|dislikes|
+-----+--------+
| NULL|    NULL|
| NULL|    NULL|
| NULL|    NULL|
| NULL|    NULL|
| NULL|    NULL|
| NULL|    NULL|
| NULL|    NULL|
| NULL|    NULL|
| NULL|    NULL|
| NULL|    NULL|
+-----+--------+
only showing top 10 rows



En este caso, podríamos asumir que aquellos valores `null` son 0 likes / dislikes. De ser así, usaremos la función `fillna(0)` para realizar la imputación.

In [7]:
df4 = df4.withColumn('likes', F.col('likes').cast('long')) \
         .withColumn('dislikes', F.col('dislikes').cast('long'))

In [8]:
df4.fillna(0).select(
    F.col('likes'), F.col('dislikes')
).orderBy('likes').show(10)

+-----+--------+
|likes|dislikes|
+-----+--------+
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
+-----+--------+
only showing top 10 rows



También lo podemos aplicar a columnas concretas usando el parámetro `subset=<[columnas]>` en la función `fillna()`.

In [9]:
df5 = df4.fillna(0, subset=['likes','dislikes'])
df5.select(
    F.col('likes'), F.col('dislikes')
).orderBy('likes').show(10)

+-----+--------+
|likes|dislikes|
+-----+--------+
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
|    0|       0|
+-----+--------+
only showing top 10 rows



### 2.8 Acciones en DataFrames

#### **Función `show()`**

Nos muestra los 20 primeros resultados del dataframe. Se le puede especificar el número de filas a mostrar con `<num columnas>`. Del mismo modo, se puede especificar el parámetro `truncate=<bool>` para mostrar todo el ancho de la columna o no.

In [10]:
df4.show(2, truncate=False)
df5.show(2,0)

df4.show(2, truncate=True)
df5.show(2,1)

+-----------+-------------+--------------------------------------------------------------+---------------+-----------+------------------------+--------------------------------------------------------------------------------------------------------+-------+-----+--------+-------------+----------------------------------------------+-----------------+----------------+----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

#### **Función `head()`**

Nos muestra las `n` primeros registros.

In [11]:
df5.head(2)

[Row(video_id='2kyS6SvSYSE', trending_date='17.14.11', title='WE WANT TO TALK ABOUT OUR MARRIAGE', channel_title='CaseyNeistat', category_id='22', publish_time='2017-11-13T17:13:01.000Z', tags='SHANtell martin', views='748374', likes=57527, dislikes=2966, comment_count='15954', thumbnail_link='https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg', comments_disabled='False', ratings_disabled='False', video_error_or_removed='False', description="SHANTELL'S CHANNEL - https://www.youtube.com/shantellmartin\\nCANDICE - https://www.lovebilly.com\\n\\nfilmed this video in 4k on this -- http://amzn.to/2sTDnRZ\\nwith this lens -- http://amzn.to/2rUJOmD\\nbig drone - http://tinyurl.com/h4ft3oy\\nOTHER GEAR ---  http://amzn.to/2o3GLX5\\nSony CAMERA http://amzn.to/2nOBmnv\\nOLD CAMERA; http://amzn.to/2o2cQBT\\nMAIN LENS; http://amzn.to/2od5gBJ\\nBIG SONY CAMERA; http://amzn.to/2nrdJRO\\nBIG Canon CAMERA; http://tinyurl.com/jn4q4vz\\nBENDY TRIPOD THING; http://tinyurl.com/gw3ylz2\\nYOU NEED THIS FOR THE 

### 2.9 Agregaciones

Spark permite diferentes niveles de agrupación de filas:

- **Tratar un DF como un grupo**.
- **Dividir un DF en grupos**, utilizando 1 o + columnas, y realizar 1 o + agregaciones en cada grupo.
- **Dividir un DF en ventanas** y realizar una media, un sumatorio o una clasificación.

In [33]:
df_v = spark.read.parquet(DATA_PATH + 'vuelos')

In [34]:
df_v.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [35]:
df_v.show(5, 0)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

In [36]:
df_d = spark.read.parquet(DATA_PATH + 'dataframe.parquet')

In [37]:
df_d.printSchema()

root
 |-- nombre: string (nullable = true)
 |-- color: string (nullable = true)
 |-- cantidad: long (nullable = true)



In [38]:
df_d.show()

+------+-----+--------+
|nombre|color|cantidad|
+------+-----+--------+
|  Jose| azul|    1900|
|  NULL| NULL|    1700|
|  NULL| rojo|    1300|
|  Juan| rojo|    1500|
+------+-----+--------+



#### **Función `count()`**

Hace recuento del número de elementos **NO NULOS** de una columna o columnas determinadas. Usando "*" podemos obtener un conteno general de las filas existentes (igual que en SQL).

In [39]:
df_d.select(
    F.count('nombre').alias('conteo_nombre'),
    F.count('color').alias('conteo_color')
).show()

+-------------+------------+
|conteo_nombre|conteo_color|
+-------------+------------+
|            2|           3|
+-------------+------------+



In [40]:
df_d.select(
    F.count('nombre').alias('conteo_nombre'),
    F.count('color').alias('conteo_color'),
    F.count('*').alias('conteo_general')
).show()

+-------------+------------+--------------+
|conteo_nombre|conteo_color|conteo_general|
+-------------+------------+--------------+
|            2|           3|             4|
+-------------+------------+--------------+



#### **Función `countDistinct()`**

Hace recuento del número de elementos **DISTINTOS NO NULOS** de una columna o columnas determinadas.

In [41]:
df_d.select(
    F.countDistinct('color').alias('colores_diferentes')
).show()

+------------------+
|colores_diferentes|
+------------------+
|                 2|
+------------------+



#### **Función `approx_count_distinct()`**

Nos da un recuento **aproximado** de valores únicos.

In [42]:
df_v.select(
    F.countDistinct('AIRLINE'),
    F.approx_count_distinct('AIRLINE')
).show()

+-----------------------+------------------------------+
|count(DISTINCT AIRLINE)|approx_count_distinct(AIRLINE)|
+-----------------------+------------------------------+
|                     14|                            13|
+-----------------------+------------------------------+



#### **Función `max() / min()`**

In [43]:
df_v.select(
    F.min('AIR_TIME').alias('menor_tiempo'),
    F.max('AIR_TIME').alias('mayor_tiempo')
).show()

+------------+------------+
|menor_tiempo|mayor_tiempo|
+------------+------------+
|           8|         649|
+------------+------------+



#### **Función `sum() / sum_distinct()`**

In [44]:
df_v.select(
    F.sum('DISTANCE').alias('suma_distancia')
).show()

+--------------+
|suma_distancia|
+--------------+
|     143565620|
+--------------+



In [45]:
df_v.select(
    F.sum_distinct('DISTANCE').alias('suma_dist')
).show()

+---------+
|suma_dist|
+---------+
|  1395134|
+---------+



#### **Función `avg()`**

In [47]:
df_v.select(
    F.avg('AIR_TIME').alias('media_aire'),
    (F.sum('AIR_TIME') / F.count('AIR_TIME')).alias('media_manual')
).show()

+------------------+------------------+
|        media_aire|      media_manual|
+------------------+------------------+
|113.38689237477338|113.38689237477338|
+------------------+------------------+



## Ejercicios

Vamos a utilizar los datos de vuelos, que hemos nombrado antes como `df_v`.

#### **Ejercicio 1 – Cargar y explorar el DataFrame**
Carga los archivos Parquet en un DataFrame de PySpark y muestra las primeras filas junto con su esquema para familiarizarte con la estructura de los datos.

**Resultado esperado:**
Muestra las columnas del dataset (por ejemplo, `year`, `month`, `day`, `airline`, `origin`, `dest`, `dep_delay`, `arr_delay`, `distance`, etc.) y las primeras filas.

#### **Ejercicio 2 – Seleccionar y filtrar vuelos**
Selecciona las columnas `year`, `month`, `airline`, `origin`, `dest`, `dep_delay` y `arr_delay`, y filtra los vuelos con retraso de salida mayor a 60 minutos.

**Resultado esperado:**
Muestra los vuelos con más de 60 minutos de retraso en salida.

#### **Ejercicio 3 – Contar valores nulos por columna**
Identifica cuántos valores nulos hay en las columnas de retrasos (`dep_delay` y `arr_delay`).

**Resultado esperado:**
Muestra el número de valores nulos en `dep_delay` y `arr_delay`.

#### **Ejercicio 4 – Imputar valores nulos**
Reemplaza los valores nulos de `dep_delay` y `arr_delay` por 0 (suponiendo que los datos faltantes se refieren a vuelos sin retraso).

**Resultado esperado:**
Los valores nulos se reemplazan por 0.

#### **Ejercicio 5 – Crear columna de retraso total**
Crea una nueva columna llamada `total_delay` que sea la suma de `dep_delay` y `arr_delay`.

**Resultado esperado:**
Aparece una nueva columna `total_delay` con la suma de ambos retrasos.

#### **Ejercicio 6 – Agrupar por aerolínea**
Calcula el promedio de retraso total (`total_delay`) por aerolínea.

**Resultado esperado:**
Una tabla con cada aerolínea y su retraso medio, ordenada de mayor a menor.


#### **Ejercicio 7 – Agrupar por origen y destino**
Cuenta cuántos vuelos hay entre cada par de aeropuertos (`origin`, `dest`).

**Resultado esperado:**
Muestra las rutas más frecuentes entre aeropuertos, con su número de vuelos.


#### **Ejercicio 8 – Conteo y countDistinct**
Calcula:
1. El número total de vuelos.
2. El número de aerolíneas únicas.
3. El número de aeropuertos de origen distintos.

**Resultado esperado:**
Devuelve tres valores: total de vuelos, número de aerolíneas y número de aeropuertos de origen únicos.

#### **Ejercicio 9 – Filtrar y ordenar vuelos retrasados**
Muestra los 10 vuelos más retrasados en llegada (`arr_delay`), mostrando aerolínea, origen, destino y minutos de retraso.

**Resultado esperado:**
Los 10 vuelos con mayor retraso en llegada.

#### **Ejercicio 10 – Agrupación múltiple y promedio mensual**
Calcula el promedio de retraso de salida (`dep_delay`) por mes y aerolínea.

**Resultado esperado:**
Una tabla con el mes, la aerolínea y su retraso medio de salida.